# 03 — Exploratory Data Analysis (EDA): User Demographics & Interview Essentials
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Machine Learning, and Analytics Interviews.*

---

## 📌 Executive Summary & Interview Expectations
In technical interviews (especially live coding and take-home data challenges), interviewers give you an unfamiliar tabular dataset (like this MovieLens `u.user` demographic dataset) to assess your **systematic EDA framework**, your **defensive coding habits**, and your awareness of **common data ingestion traps**.

### Core Competencies Tested in this Module:
1. **Delimited File Ingestion**: Custom delimiters (`sep='|'`), header detection, and index assignment.
2. **Dataset Dimensionality & Shape**: Extracting observation and feature counts idiomatic to Python/Pandas.
3. **Column Access Best Practices**: Why bracket notation `df['col']` is strictly preferred over dot notation `df.col`.
4. **Summary Statistics (`describe`)**: Understanding numeric vs categorical summaries (`count`, `unique`, `top`, `freq`).
5. **Cardinality & Frequency Distributions**: `.nunique()`, `.value_counts()`, finding modes, and identifying rare edge-case categories.
6. **Data Type Traps**: Why ZIP codes and identifiers must **never** be parsed as integers.
7. **Interview Corner**: Handling multi-modal ties, demographic ratio calculations, and method chaining.

## 1. Environment Setup & Ingestion
The dataset uses pipe (`|`) delimiters instead of standard commas.

> 💡 **Interview Note — Ingestion Edge Cases**:
> - Always specify `sep='|'` (or `delimiter='|'`).
> - Use `index_col='user_id'` when an unambiguous primary key is provided.
> - **Zip Code Gotcha**: Notice `zip_code` contains alphanumeric strings and leading zeros (e.g. US east coast ZIP codes start with `0`). Parsing it as an integer strips leading zeros!

In [1]:
import os
import numpy as np
import pandas as pd

# Robust loader: load local user.csv or fall back to public repository
file_path = "user.csv"
if not os.path.exists(file_path):
    file_path = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/u.user"

users = pd.read_csv(file_path, sep="|", index_col="user_id")
print("Data loaded successfully. Dataset dimensions:", users.shape)

Data loaded successfully. Dataset dimensions: (943, 4)


## 2. Inspecting Observations: `head()` and `tail()`
Inspect the top 25 and bottom 10 entries to verify record formatting, missing value conventions, and schema integrity.

In [2]:
# First 25 records
users.head(25)

,age,gender,occupation,zip_code
user_id,,,,
1,24,M,technician,85711
2,53,F,other,94043
3,23,M,writer,32067
4,24,M,technician,43537
5,33,F,other,15213
6,42,M,executive,98101
7,57,M,administrator,91344
8,36,M,administrator,05201
9,29,M,student,01002


In [3]:
# Last 10 records
users.tail(10)

,age,gender,occupation,zip_code
user_id,,,,
934,61,M,engineer,22902
935,42,M,doctor,66221
936,24,M,other,32789
937,48,M,educator,98072
938,38,F,technician,55038
939,26,F,student,33319
940,32,M,administrator,02215
941,20,M,student,97229
942,48,F,librarian,78209


## 3. Dimensionality, Columns, and Index Inspection

### ⚠️ Top Interview Question: Accessing Shape Elements
- Number of observations (rows): `users.shape[0]` or `len(users)`.
- Number of features (columns): `users.shape[1]` or `len(users.columns)`.
- Index inspection: `users.index` shows the primary key index (`Int64Index` or `Index`).

In [4]:
print(f"Number of observations (rows):    {users.shape[0]}")
print(f"Number of columns (features):      {users.shape[1]}")
print(f"Dataset Index:                     {users.index}")
print(f"Column Names:                      {list(users.columns)}")

Number of observations (rows):    943
Number of columns (features):      4
Dataset Index:                     RangeIndex(start=1, stop=944, step=1, name='user_id')
Column Names:                      ['age', 'gender', 'occupation', 'zip_code']


## 4. Column Inspection: Bracket Notation vs Dot Notation

### ⚠️ Top Interview Trap: `df.column` vs `df['column']`
In casual code, developers often type `users.occupation`. In production and technical interviews, **always use bracket notation `users['occupation']`** because:
1. **Name Clashes**: If a column name matches an existing DataFrame attribute or method (e.g. `count`, `size`, `shape`, `min`, `max`, `values`), dot notation accesses the *method*, NOT the column!
2. **Special Characters & Spaces**: Dot notation fails if column names have spaces (`users.first name`) or special characters.
3. **Dynamic Variable Access**: `users[col_var]` works dynamically in loops and functions; `users.col_var` looks for a literal column named `col_var`.
4. **Assignment / New Columns**: Creating a new column `df.new_col = ...` fails to add it to the DataFrame columns dictionary; only `df['new_col'] = ...` creates it correctly!

In [5]:
# Check data types of all features
users.dtypes

age           int64
gender          str
occupation      str
zip_code        str
dtype: object

In [6]:
# Accessing a single column (using robust bracket notation)
users["occupation"].head()

user_id
1    technician
2         other
3        writer
4    technician
5         other
Name: occupation, dtype: str

## 5. Categorical Feature Exploration: Unique Values & Frequencies

> 💡 **Interview Distinction**:
> - `.unique()`: Returns an **array of unique values** (including `NaN`).
> - `.nunique()`: Returns the **count of unique values** (skips `NaN` by default; set `dropna=False` to count nulls as a distinct category).

In [7]:
num_occupations = users["occupation"].nunique()
print(f"Number of distinct occupations: {num_occupations}")

Number of distinct occupations: 21


### 💡 Identifying the Most Frequent Category (Mode)
Two common methods in interviews:
1. `users['occupation'].value_counts().idxmax()`: Returns the single label with the highest count (efficient, but picks the first in case of a tie).
2. `users['occupation'].mode()[0]`: Formal statistical mode (returns all ties if multi-modal).

In [8]:
# Most frequent occupation via value_counts
top_occupation = users["occupation"].value_counts().idxmax()
top_count = users["occupation"].value_counts().max()

print(f"Most frequent occupation: '{top_occupation}' with {top_count} users")

Most frequent occupation: 'student' with 196 users


## 6. Comprehensive Statistical Summaries: `describe()`

### 💡 Interview Deep-Dive: The `include` Parameter of `describe()`
- Default `df.describe()`: Only analyzes **numeric columns** (`age`).
- `df.describe(include="all")`: Analyzes **both numeric and categorical** columns.
  - For categorical columns, it introduces:
    - `unique`: Number of distinct classes.
    - `top`: Most common category (the mode).
    - `freq`: Count/frequency of the most common category.
- `df.describe(include="object")` or `include=["category"]`: Filters summary strictly to qualitative data.

In [9]:
# Numeric-only summary
users.describe()

,age
count,943.000000
mean,34.051962
std,12.192740
min,7.000000
25%,25.000000
50%,31.000000
75%,43.000000
max,73.000000


In [10]:
# Summary across all columns (numeric + categorical)
users.describe(include="all")

,age,gender,occupation,zip_code
count,943.000000,943,943,943
unique,NaN,2,21,795
top,NaN,M,student,55414
freq,NaN,670,196,9
mean,34.051962,NaN,NaN,NaN
std,12.192740,NaN,NaN,NaN
min,7.000000,NaN,NaN,NaN
25%,25.000000,NaN,NaN,NaN
50%,31.000000,NaN,NaN,NaN
75%,43.000000,NaN,NaN,NaN


In [11]:
# Summarize single categorical series
users["occupation"].describe()

count         943
unique         21
top       student
freq          196
Name: occupation, dtype: object

## 7. Demographic Distributions & Rare Value Detection

Detecting rare values (the tail of `value_counts()`) is a standard interview data-quality question to identify rare edge cases or data-entry typos.

In [12]:
# Mean user age
mean_age = users["age"].mean()
print(f"Mean user age: {mean_age:.2f} years (Rounded: {round(mean_age)})")

Mean user age: 34.05 years (Rounded: 34)


In [13]:
# Least occurring ages in the dataset
print("Ages with the fewest occurrences:")
users["age"].value_counts().tail(10)

Ages with the fewest occurrences:


age
70    3
62    2
68    2
64    2
69    2
7     1
66    1
11    1
10    1
73    1
Name: count, dtype: int64

## 8. EDA Quick Reference Cheat Sheet

| Question / Task | Idiomatic Pandas Expression | Key Consideration |
| :--- | :--- | :--- |
| **Row Count** | `df.shape[0]` or `len(df)` | $O(1)$ time complexity |
| **Column Count** | `df.shape[1]` or `len(df.columns)` | Tuple indexing on shape |
| **Column Selection** | `df['col']` | Avoid `df.col` to prevent method name collisions |
| **Cardinality** | `s.nunique(dropna=False)` | Mention `dropna=False` in interviews |
| **Most Frequent (Mode)** | `s.value_counts().idxmax()` or `s.mode()[0]` | `mode()` returns all ties |
| **Full Summary** | `df.describe(include='all')` | Displays `top` and `freq` for object columns |
| **Infrequent Values** | `s.value_counts().tail()` | Ideal for identifying outliers / typos |

---
## 🎯 9. Technical Interview Corner: Tricky Questions & Drills

### Q1: The Dot Notation Disaster
**Question**: A candidate wrote the following code to count occurrences of values in a column named `count`:
```python
df.count.value_counts()
```
Why did this raise an `AttributeError` or fail completely, and what is the fix?

**Answer**:
`df.count` accesses the built-in DataFrame method `DataFrame.count()` (which computes non-null values for each column), NOT the column named `'count'`. Attempting to call `.value_counts()` on a bound method object results in:
`AttributeError: 'function' object has no attribute 'value_counts'`.
**Fix**: Always use bracket notation: `df['count'].value_counts()`.

In [14]:
# Demonstration of method name collision
demo = pd.DataFrame({"count": [10, 20, 10, 30], "shape": ["A", "B", "A", "C"]})

print("demo['count'] (Correct):")
print(demo["count"].value_counts())

print("\nType of demo.count (Built-in method, NOT column!):")
print(type(demo.count))

demo['count'] (Correct):
count
10    2
20    1
30    1
Name: count, dtype: int64

Type of demo.count (Built-in method, NOT column!):
<class 'method'>


### Q2: Why Postal Codes & Identifiers Must NEVER Be Integers
**Question**: In this dataset, `zip_code` is stored as an `object` (string). An interviewer asks: *"Why don't we convert `zip_code` and `user_id` to `int` to save memory?"* How should you respond?

**Answer**:
1. **Leading Zeros**: US East Coast ZIP codes begin with `0` (e.g. `07030` Hoboken, NJ). Converting to integer truncates them to `7030`, causing silent data corruption!
2. **Alphanumeric Codes**: Many countries (UK, Canada) have alphanumeric postal codes (e.g., `K1A 0B1`).
3. **Semantic Integrity**: Identifiers and postal codes are categorical labels, not quantities. You will never perform arithmetic (mean, sum, division) on a zip code or user ID. Storing them as integers invites accidental arithmetic errors.

In [15]:
# Demonstrating the leading zero truncation bug
raw_zip = "02138"  # Cambridge, MA
int_zip = int(raw_zip)
print(f"Original ZIP: '{raw_zip}' -> As Integer: {int_zip} (CORRUPTED!)")

Original ZIP: '02138' -> As Integer: 2138 (CORRUPTED!)


### Q3: Multi-modal Distributions & Mode Ties
**Question**: What happens if two occupations have the exact same highest frequency in `value_counts()`? Does `.idxmax()` return both?

**Answer**:
No. `.idxmax()` returns only the **first** label encountered that achieved the maximum value.
If you need to correctly detect all tied modes in an interview, use `df['col'].mode()` which returns a `Series` containing all values with maximal frequency.

In [16]:
tied_series = pd.Series(["student", "engineer", "student", "engineer", "doctor"])
print("value_counts:")
print(tied_series.value_counts())

print("\n.idxmax() (Returns ONLY ONE):", tied_series.value_counts().idxmax())
print(".mode() (Returns ALL TIES):     ", list(tied_series.mode()))

value_counts:
student     2
engineer    2
doctor      1
Name: count, dtype: int64

.idxmax() (Returns ONLY ONE): student
.mode() (Returns ALL TIES):      ['engineer', 'student']


### Q4: Hands-on Interview Coding Challenge: Gender Ratio by Occupation
**Challenge**: In a single chained expression, find the **top 3 occupations with the highest ratio of male users to female users**, considering only occupations that have **at least 20 total users**.

In [17]:
# Solution to Coding Challenge using cross-tabulation and method chaining
gender_ratio = (
    pd.crosstab(users["occupation"], users["gender"])
    .assign(total=lambda df: df["M"] + df["F"])
    .query("total >= 20")
    .assign(male_to_female_ratio=lambda df: (df["M"] / df["F"]).round(2))
    .sort_values(by="male_to_female_ratio", ascending=False)
    .head(3)
)

gender_ratio

gender,F,M,total,male_to_female_ratio
occupation,,,,
engineer,2,65,67,32.5
technician,1,26,27,26.0
programmer,6,60,66,10.0
